In [0]:
import sys

sys.path.append("/Workspace/Users/sjaddya98@gmail.com/energy-lakehouse/src")

In [0]:
from energy_lakehouse.tables import CATALOG, SCHEMAS

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
for schema in SCHEMAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    print("ready:", schema)

In [0]:
from energy_lakehouse.tables import DWD_RAW, SMARD_RAW

LEGACY = {
    SMARD_RAW: "workspace.bronze.smard_raw",
    DWD_RAW: "workspace.bronze.dwd_raw",
}

for new, old in LEGACY.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {new} DEEP CLONE {old}")
    print(f"{old} -> {new}: {spark.table(new).count()} rows")

In [0]:
%sql
SELECT 'smard_raw' AS tbl,
       (SELECT count(*)           FROM workspace.bronze.smard_raw) AS old_rows,
       (SELECT count(*)           FROM energy.bronze.smard_raw)    AS new_rows,
       (SELECT count(value)       FROM workspace.bronze.smard_raw) AS old_values,
       (SELECT count(value)       FROM energy.bronze.smard_raw)    AS new_values,
       (SELECT max(timestamp_utc) FROM workspace.bronze.smard_raw) AS old_last,
       (SELECT max(timestamp_utc) FROM energy.bronze.smard_raw)    AS new_last
UNION ALL
SELECT 'dwd_raw',
       (SELECT count(*)           FROM workspace.bronze.dwd_raw),
       (SELECT count(*)           FROM energy.bronze.dwd_raw),
       (SELECT count(value)       FROM workspace.bronze.dwd_raw),
       (SELECT count(value)       FROM energy.bronze.dwd_raw),
       (SELECT max(timestamp_utc) FROM workspace.bronze.dwd_raw),
       (SELECT max(timestamp_utc) FROM energy.bronze.dwd_raw)

In [0]:
%sql
SELECT
  (SELECT count(*) FROM (SELECT filter_id, timestamp_utc
                         FROM energy.bronze.smard_raw
                         GROUP BY filter_id, timestamp_utc HAVING count(*) > 1)) AS smard_duplicate_keys,
  (SELECT count(*) FROM (SELECT station_id, timestamp_utc, parameter
                         FROM energy.bronze.dwd_raw
                         GROUP BY station_id, timestamp_utc, parameter HAVING count(*) > 1)) AS dwd_duplicate_keys